# 01 · 전부 넣기 vs 골라 넣기
**1세션 · 컨텍스트란 무엇인가** — 실습 20분

같은 질문을 두 조건으로 보내고 **정답 여부 · 응답 시간 · 입력 토큰 · 비용** 네 가지를 측정합니다.

| 조건 | 답이 맞았나 | 응답 시간 | 입력 토큰 | 1건 비용 |
|---|---|---|---|---|
| A 사내 자료 20편 전부 | | | | |
| B 관련 부분만 | | | | |

A가 더 정확하다면 그것도 결과입니다. 중요한 것은 **차이를 숫자로 적어 보는 것**입니다.

> 요즘 모델은 창이 커서(수십만~백만 토큰) "그냥 다 넣으면 되지 않나" 싶을 수 있습니다. 창에 들어간다고 끝이 아닙니다 — ①보낸 토큰만큼 비용·시간이 **창 크기와 무관하게** 그대로 붙고, ②관련 없는 내용이 많을수록 모델이 정답을 놓치는 경우가 보고돼 있습니다("context rot"). 이 둘을 지금 두 조건으로 직접 측정해 봅니다.

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
print("모델:", llm.MODEL, "| 모의 모드" if llm.MOCK else "| 실제 호출")

## 1. 창에 들어갈 재료
가상의 회사 **한빛정밀**의 사내 자료 20편(`data/kb/`)입니다. 전부 합치면 몇 토큰일까요?

In [ ]:
docs = sorted(Path("data/kb").glob("*.txt"))
ALL = "\n\n".join(f"<문서 출처=\"{p.name}\">\n{p.read_text(encoding='utf-8')}\n</문서>" for p in docs)
for p in docs:
    print(f"{p.name:<26} {len(p.read_text(encoding='utf-8')):>5}자")
print(f"\n합계 {len(ALL):,}자 → {llm.count_tokens(ALL):,} 토큰")

## 2. 질문 고르기
답이 **문서 가운데쯤**에 있고, **헷갈리게 하는 비슷한 내용**이 다른 문서에 있는 질문을 고릅니다.

- Q1 경조휴가 — 답은 일반 업무 규칙 29개 조 중 제16조. 복리후생 안내에 *경조금*이 따로 있다
- Q2 AX-2041 토크 — 이름이 비슷한 AX-2014 사양서가 따로 있다

In [ ]:
QUESTIONS = [
    {"q": "직원 본인이 결혼하면 경조휴가는 며칠인가요? 숫자와 근거 조항만 답하세요.",
     "answer": "5일",
     "part": Path("data/kb/01_일반업무규칙.txt").read_text(encoding="utf-8").split("제16조")[1].split("제17조")[0]},
    {"q": "AX-2041의 정격 토크와 최대 토크는 각각 얼마인가요? 숫자만 답하세요.",
     "answer": "4.5",
     "part": Path("data/kb/06_제품사양_AX-2041.txt").read_text(encoding="utf-8")},
]
print("B 조건에 넣을 부분 (Q1):\n제16조" + QUESTIONS[0]["part"])

## 3. A·B 두 조건으로 같은 질문을 보내고 측정한다
각 질문을 조건 A(자료 전부)·B(관련 부분만)로 각각 한 번씩 보내, 정답 여부 · 응답 시간 · 입력 토큰 · 비용을 표에 기록합니다.

In [ ]:
def measure(context, q):
    prompt = f"<자료>\n{context}\n</자료>\n\n<질문>{q}</질문>"
    r = llm.call(prompt, max_tokens=300)
    return r

table = []
for item in QUESTIONS:
    for name, ctx in [("A 전부", ALL), ("B 골라서", item["part"])]:
        r = measure(ctx, item["q"])
        ok = item["answer"] in r.text
        table.append((item["q"][:18], name, "○" if ok else "×", r.seconds, r.input_tokens, llm.fmt_cost(r.cost)))
        print(f"[{name}] {r.text.strip()[:80]}")
    print()

print(f"{'질문':<20}{'조건':<10}{'정답':<5}{'시간':>7}{'입력 토큰':>10}  비용")
for q, name, ok, sec, tin, c in table:
    print(f"{q:<20}{name:<10}{ok:<5}{sec:>6.1f}초{tin:>10,}  {c}")

## 4. 세 번씩 돌려 본다
한 번은 우연일 수 있습니다 — 어제 배운 것. A 조건을 세 번 더 돌려 매번 맞는지 봅니다.

In [ ]:
for item in QUESTIONS:
    hits = sum(item["answer"] in measure(ALL, item["q"]).text for _ in range(3))
    print(f"A 전부 · {item['q'][:20]} → 3번 중 {hits}번 정답")

## 관찰 기록
- 입력 토큰은 몇 배 차이 났나:
- 응답 시간은:
- 정답은 둘 다 맞았나. 틀렸다면 무엇과 헷갈렸나 (경조금? AX-2014?):
- 하루 1,000번 묻는 사내 도우미라면, A와 B의 한 달 비용 차이는:

> 오늘의 결론으로 이어집니다 — *창에 넣는 것과 모델이 잘 쓰는 것은 다르다.* 그래서 **골라 넣는 방법(RAG)**이 3세션의 주제입니다.